In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

from students.kirienko.lesson3 import Exercise

# Загрузка данных
digits = load_digits()
X, y = digits.data.astype(np.float32), digits.target
X = X / 16.0
print(f"Всего образцов: {len(X)}, Признаков: {X.shape[1]}, Классов: {len(np.unique(y))}")

# Разделение данных
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)
print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")


# Функция обучения
def evaluate_hyperparameters(
    n_neurons: int,
    lr: float,
    n_epoch: int,
    batch_size: int,
) -> float:
    rng = np.random.default_rng(42)
    model = Exercise.create_model(
        Exercise.create_linear_layer(64, n_neurons, rng),
        Exercise.create_relu_layer(),
        Exercise.create_linear_layer(n_neurons, 10, rng),
        Exercise.create_logsoftmax_layer(),
    )
    loss_fn = Exercise.create_nll_loss()
    Exercise.train_model(model, loss_fn, X_train, y_train, lr, n_epoch, batch_size)
    val_log_probs = model.forward(X_val)
    preds = np.argmax(val_log_probs, axis=1)
    return float(np.mean(preds == y_val))


print(" Случайный поиск гиперпараметров (100 попыток)")

best_acc = 0.0
best_params: dict[str, int | float] = {}
results: list[tuple[float, dict[str, int | float]]] = []
rng_search = np.random.default_rng(123)

for i in range(100):
    # Явное преобразование numpy типов в Python типы
    params: dict[str, int | float] = {
        "n_neurons": int(rng_search.integers(32, 512)),
        "lr": float(10 ** rng_search.uniform(-4, -1)),
        "n_epoch": int(rng_search.integers(10, 100)),
        "batch_size": int(rng_search.integers(8, 256)),
    }

    acc = evaluate_hyperparameters(
        n_neurons=params["n_neurons"],
        lr=params["lr"],
        n_epoch=params["n_epoch"],
        batch_size=params["batch_size"],
    )
    results.append((acc, params))

    if acc > best_acc:
        best_acc = acc
        best_params = params.copy()
        print(f"#{i + 1:3d}  NEW BEST: {acc:.4f} | {params}")
    elif i % 20 == 0:
        print(f"#{i + 1:3d}    Acc: {acc:.4f}")

# Результаты
print(" Топ-5 лучших конфигураций:")

results.sort(key=lambda x: -x[0])
for rank, (acc, params) in enumerate(results[:5], 1):
    print(
        f"{rank}. {acc:.4f} | neurons={params['n_neurons']}, "
        f"lr={params['lr']:.5f}, epochs={params['n_epoch']}, "
        f"batch={params['batch_size']}"
    )

print(f"\n Лучшая точность на валидации: {best_acc:.4f}")
print(f" Лучшие параметры: {best_params}")

# Финальное обучение
print(" Финальное обучение на объединённых данных")

X_train_val = np.vstack([X_train, X_val])
y_train_val = np.hstack([y_train, y_val])

rng_final = np.random.default_rng(42)
final_model = Exercise.create_model(
    Exercise.create_linear_layer(64, int(best_params["n_neurons"]), rng_final),
    Exercise.create_relu_layer(),
    Exercise.create_linear_layer(int(best_params["n_neurons"]), 10, rng_final),
    Exercise.create_logsoftmax_layer(),
)
loss_fn = Exercise.create_nll_loss()

Exercise.train_model(
    final_model,
    loss_fn,
    X_train_val,
    y_train_val,
    float(best_params["lr"]),
    int(best_params["n_epoch"]),
    int(best_params["batch_size"]),
)


# Оценка на тесте
test_log_probs = final_model.forward(X_test)
preds = np.argmax(test_log_probs, axis=1)
test_acc = float(np.mean(preds == y_test))

print(f"\n Test accuracy: {test_acc:.4f}")
print(f" Val accuracy:  {best_acc:.4f}")
print(f" Gap: {abs(test_acc - best_acc):.4f}")

# Матрица ошибок
print(" Матрица ошибок:")

cm: np.ndarray = np.zeros((10, 10), dtype=np.int32)
for true, pred in zip(y_test, preds, strict=True):
    cm[true, pred] += 1

cm_df = pd.DataFrame(
    cm,
    index=[f"True_{i}" for i in range(10)],
    columns=[f"Pred_{i}" for i in range(10)],
)
print(cm_df)

# Анализ по классам
print(" Анализ по классам:")

for digit in range(10):
    total = int(cm[digit].sum())
    correct = int(cm[digit, digit])
    acc_class = correct / total if total > 0 else 0.0
    print(f"Цифра {digit}: {correct}/{total} правильно ({acc_class * 100:.1f}%)")